# Notebook 02 — Baseline MC Evaluation
**AITR_Lab02_RushikeshShirsath_LLMFactuality**

Evaluates Llama-3.1-8B-Instruct and Mistral-7B-Instruct-v0.3 on the **multiple-choice** format of PubMedQA.

Run for all 4 conditions: {llama, mistral} × {no-RAG, BM25-RAG}

**Pre-condition:** Notebook 01 must pass all checks first.

**Expected time:** ~2 hours per model-condition on Colab T4.

In [ ]:
# Cell 1: Setup
import sys, os, json, random, time
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

sys.path.insert(0, '/content')

from src.data_loader import load_pubmedqa_test
from src.evaluate import parse_mc_answer, compute_mc_metrics
from src.prompt_templates import MC_TEMPLATE, format_context
from src.rag import build_bm25_index, retrieve

# Freeze random seed (matches experiment_config.yaml)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

os.makedirs('results', exist_ok=True)
print(f'CUDA: {torch.cuda.is_available()} | Seed: {SEED}')

In [ ]:
# Cell 2: Load test data
test_items = load_pubmedqa_test(n=500)
print(f'Test items loaded: {len(test_items)}')

# Majority-class baseline (always predict 'yes')
majority_preds  = ['yes'] * len(test_items)
majority_labels = [item['final_decision'] for item in test_items]
majority_metrics = compute_mc_metrics(majority_preds, majority_labels)
print(f'\nMajority-class baseline:')
print(f'  Accuracy: {majority_metrics["accuracy"]:.4f}  (expected ~0.5500 — VERIFIED)')
print(f'  Macro-F1: {majority_metrics["macro_f1"]:.4f}')

In [ ]:
# Cell 3: Model and RAG configuration
# CHANGE THESE TWO LINES to switch model/condition:
MODEL_NAME = 'mistral'          # 'mistral' or 'llama'
USE_RAG    = False              # False or True

MODEL_IDS = {
    'llama':   'meta-llama/Llama-3.1-8B-Instruct',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
}
FALLBACK_IDS = {
    'llama':   'BioMistral/BioMistral-7B',
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
}
RAG_TAG = 'rag' if USE_RAG else 'norag'
print(f'Config: model={MODEL_NAME}, rag={USE_RAG}, tag={RAG_TAG}')

In [ ]:
# Cell 4: HuggingFace login (required for Llama only)
if MODEL_NAME == 'llama':
    from huggingface_hub import login
    # Set your token here OR use: !huggingface-cli login
    HF_TOKEN = 'your_hf_token_here'  # REPLACE — do NOT commit this
    login(token=HF_TOKEN)
    print('Logged in to HuggingFace.')
else:
    print('Mistral-7B is open — no login required.')

In [ ]:
# Cell 5: Load model
hf_id = MODEL_IDS[MODEL_NAME]
print(f'Loading: {hf_id}')
try:
    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForCausalLM.from_pretrained(
        hf_id, load_in_8bit=True, device_map='auto')
except Exception as e:
    print(f'Failed: {e}\nUsing fallback: {FALLBACK_IDS[MODEL_NAME]}')
    hf_id = FALLBACK_IDS[MODEL_NAME]
    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    model = AutoModelForCausalLM.from_pretrained(
        hf_id, load_in_8bit=True, device_map='auto')
model.eval()
print(f'Model loaded: {hf_id}')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# Cell 6: Build BM25 index (if RAG enabled)
bm25, doc_texts = None, None
if USE_RAG:
    print('Building BM25 index...')
    bm25, doc_texts = build_bm25_index(n_test=500)
    print(f'Index ready: {len(doc_texts)} docs')
else:
    print('RAG disabled — using parametric model only.')

In [ ]:
# Cell 7: Run MC evaluation
def generate_mc(prompt):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                       max_length=1024).to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=5, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

mc_preds, mc_labels, rows = [], [], []
start = time.time()

for i, item in enumerate(test_items):
    q       = item['question']
    ctx     = format_context(item.get('context', {}), max_words=400)
    gold_mc = item['final_decision']

    if USE_RAG:
        retrieved = retrieve(bm25, doc_texts, q, top_k=1)
        ctx = (retrieved + ' ' + ctx)[:800]
    else:
        ctx = ctx[:800]

    prompt  = MC_TEMPLATE.format(question=q, context=ctx)
    raw_out = generate_mc(prompt)
    pred_mc = parse_mc_answer(raw_out)

    mc_preds.append(pred_mc)
    mc_labels.append(gold_mc)
    rows.append({'item_id': i, 'question': q[:80], 'gold_mc': gold_mc,
                 'pred_mc': pred_mc, 'raw_output': raw_out[:30],
                 'correct': int(pred_mc == gold_mc)})

    if (i + 1) % 10 == 0:
        acc = sum(r['correct'] for r in rows) / len(rows)
        print(f'  [{i+1}/500] Running acc: {acc:.3f} | {time.time()-start:.0f}s')

    if (i + 1) % 50 == 0:
        with open(f'results/ckpt_mc_{MODEL_NAME}_{RAG_TAG}_{i+1}.json', 'w') as f:
            json.dump(rows, f)

print('\nEvaluation complete.')

In [ ]:
# Cell 8: Compute and display final metrics
metrics = compute_mc_metrics(mc_preds, mc_labels)
print('\n' + '='*55)
print(f'  FINAL MC RESULTS — {MODEL_NAME.upper()} | RAG={USE_RAG}')
print('='*55)
print(f'  MC Accuracy:  {metrics["accuracy"]:.4f}  (majority: 0.5500)')
print(f'  Macro-F1:     {metrics["macro_f1"]:.4f}')
print(f'  N items:      {metrics["n_items"]}')
print('='*55)

# Save results
df = pd.DataFrame(rows)
out_path = f'results/mc_results_{MODEL_NAME}_{RAG_TAG}.csv'
df.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')

summary = {'model': MODEL_NAME, 'rag': RAG_TAG, **metrics}
with open(f'results/summary_mc_{MODEL_NAME}_{RAG_TAG}.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('\n>>> Copy the FINAL MC RESULTS above into Table 1 of paper/main.tex')